# 01 — Datos de Text-to-SQL

Turno de Jorge Lopez. Implementación y revisión técnica asistidas por IA. Incluye entrenamiento sintético y cuarenta preguntas recibidas de los integrantes; se preservan los originales y autores declarados. No incluye baseline ni entrenamiento del LLM.

In [1]:
from pathlib import Path
from collections import Counter
import os, json, subprocess, sys
raiz = Path.cwd()
if not (raiz / "scripts").exists() and (raiz.parent / "scripts").exists():
    raiz = raiz.parent
assert (raiz / "scripts").exists(), "Abrir desde el checkout del repositorio"
os.chdir(raiz)
print("Python:", sys.version.split()[0])
def jsonl(ruta):
    return [json.loads(l) for l in Path(ruta).read_text().splitlines() if l.strip()]

Python: 3.13.12


## Preparar MySQL

Seguir el README para crear el entorno e instalar requirements.txt. Con Docker activo, copiar .env.example a .env y ejecutar `docker compose up -d --wait mysql`. El servidor debe ser exclusivo de desarrollo. En Colab se requiere un servidor MySQL 8.4.10 y variables de conexión; no se afirma verificación del modelo en T4 todavía.

## Inspeccionar fuentes

Los archivos ya están incluidos. `scripts.generar_datos` y `scripts.preparar_validacion` regeneran fuentes sintéticas con sus opciones explícitas y nunca reemplazan preguntas_propias.jsonl. No entrenar con los dos esquemas reservados.

In [2]:
particiones = json.loads(Path("datos/particiones.json").read_text())
train = jsonl("datos/entrenamiento/pares.jsonl")
propias = jsonl("datos/validacion/preguntas_propias.jsonl")
assert not set(particiones["entrenamiento"]) & set(particiones["validacion"])
print("Esquemas train / val:", len(particiones["entrenamiento"]), len(particiones["validacion"]))
print("Pares train:", len(train), "familias:", len({r["familia"] for r in train}))
print("Niveles train:", dict(Counter(r["nivel"] for r in train)))
print("Preguntas propias:", len(propias), "autores:", dict(Counter(r["autor"] for r in propias)))
print("Niveles val:", dict(Counter(r["nivel"] for r in propias)), "peso:", sum(r["peso"] for r in propias))

Esquemas train / val: 15 2
Pares train: 3400 familias: 66
Niveles train: {'facil': 940, 'media': 1400, 'dificil': 1060}
Preguntas propias: 40 autores: {'Jorge Lopez': 20, 'Angel Herrarte': 20}
Niveles val: {'facil': 14, 'media': 16, 'dificil': 10} peso: 76


## Certificar y revisar

En terminal, primera ejecución: `python -m scripts.validar_datos --inicializar`. Si las bases ya existen, omitir --inicializar. Para repetir la revisión: `python -m scripts.revisar_muestra`. El comando conserva su nombre histórico pero ahora contrasta los 3,440 resultados con cálculos Python independientes. Cambios en DDL/semillas requieren reconstruir únicamente las bases sintéticas con --inicializar --recrear. Este notebook no reconstruye bases automáticamente.

In [3]:
reporte = json.loads(Path("resultados/reporte_datos.json").read_text())
print(json.dumps({k:v for k,v in reporte.items() if k != "sha256"}, ensure_ascii=False, indent=2))
revision = json.loads(Path("resultados/revision_semantica.json").read_text())
print("Resultados revisados / correctos:", revision["muestra"], revision["correctos"])
assert revision["muestra"] == revision["correctos"] == len(train) + len(propias)

{
  "fecha_utc": "2026-10-22T06:41:49.694484+00:00",
  "mysql_version": "8.4.10",
  "sql_mode": "ONLY_FULL_GROUP_BY,STRICT_TRANS_TABLES,NO_ZERO_IN_DATE,NO_ZERO_DATE,ERROR_FOR_DIVISION_BY_ZERO,NO_ENGINE_SUBSTITUTION",
  "candidatos": 3400,
  "aceptados_entrenamiento": 3400,
  "descartados_entrenamiento": 0,
  "motivos_descartes": {},
  "por_esquema": {
    "train_tienda": 240,
    "train_cafeteria": 240,
    "train_ferreteria": 240,
    "train_farmacia": 240,
    "train_libreria": 240,
    "train_repuestos": 240,
    "train_floreria": 240,
    "train_deportes": 240,
    "train_electronica": 240,
    "train_panaderia": 240,
    "train_academia": 200,
    "train_logistica": 200,
    "train_turismo": 200,
    "train_energia": 200,
    "train_manufactura": 200
  },
  "por_nivel": {
    "facil": 940,
    "media": 1400,
    "dificil": 1060
  },
  "familias_plantillas": 66,
  "resultados_vacios": 209,
  "validacion_por_nivel": {
    "facil": 14,
    "media": 16,
    "dificil": 10
  },
  "valid

## Preguntas propias

Leer `datos/validacion/PREGUNTAS.md` para ver originales, precisiones y SQL. Las preguntas 24 y 26 tienen ahora casos positivos y negativos. Los promedios incluyen ceros y los agrupamientos usan identificadores para distinguir nombres iguales. Esta revisión automática no sustituye la responsabilidad de los integrantes ni prueba generalización.

In [4]:
referencias = {r["id"]:r for r in jsonl("datos/validacion/referencias_resultados.jsonl")}
for numero in (24,26):
    p = next(r for r in propias if r["numero"] == numero)
    print(numero, p["pregunta"])
    print(referencias[p["id"]]["filas"])

24 ¿Qué citas médicas están programadas para después del 1 de enero de 2026? Devuelve el identificador de la cita, el identificador del paciente y la fecha. Considera únicamente citas pendientes, por tratarse de citas programadas.
[[77, 2, '2026-01-02']]
26 ¿Qué pacientes han tenido más de tres consultas médicas? Devuelve el nombre del paciente y la cantidad de consultas. Una consulta realizada corresponde a una cita con estado atendida.
[['Persona 01', 4], ['Persona 01', 4]]


## Auditoría de cierre

Debe fallar si faltan mínimos, evidencia, hashes vigentes o cobertura completa de revisión semántica. No certifica permisos de GitHub ni resultados de un modelo que todavía no se entrenó.

In [5]:
comando = [sys.executable, "-m", "scripts.auditar_datos", "--exigir-cierre"]
r = subprocess.run(comando, capture_output=True, text=True)
print(r.stdout)
assert r.returncode == 0, r.stderr

{
  "candidatos": 3400,
  "pares_train_certificados": 3400,
  "esquemas_train": 15,
  "esquemas_val": 2,
  "preguntas_propias": 40,
  "niveles_validacion": {
    "facil": 14,
    "media": 16,
    "dificil": 10
  },
  "autores_declarados": {
    "Jorge Lopez": 20,
    "Angel Herrarte": 20
  },
  "peso_total_validacion": 76,
  "etapa_1_completa": true,
  "topologias_entrenamiento": 6,
  "pendiente": []
}

